# Exploration IRIS + Filosofi (étape 4)

Sources (`scripts/05_download_socio.py`) :
- Filosofi 2021 par IRIS, revenus disponibles — géographie au 1er janvier 2022 (INSEE, page 8229323).
- Contours IRIS édition 2022-01-01 (IGN) — même géographie.

QPV 2024 : non téléchargé (`static.data.gouv.fr` injoignable).

In [1]:
import glob, geopandas as gpd, pandas as pd
shp = glob.glob('../data/raw/iris/**/*LAMB93_FXX*/CONTOURS-IRIS.shp', recursive=True)[0]
filtre = " OR ".join(f"INSEE_COM LIKE '{d}%'" for d in ['75', '92', '93', '94'])
iris = gpd.read_file(shp, where=filtre).drop(columns='IRIS')
print(iris.crs.to_string(), '→ équivalent EPSG:2154, à réaffecter explicitement')
iris = iris.set_crs(2154, allow_override=True)
iris['dep'] = iris.INSEE_COM.str[:2]
iris.shape

IGNF:LAMB93 → équivalent EPSG:2154, à réaffecter explicitement


(2752, 7)

## 1. Contours IRIS

Types : H = habitat, A = activité, D = divers (parcs, bois…), Z = commune non découpée en IRIS.

In [2]:
print('CODE_IRIS unique :', iris.CODE_IRIS.is_unique, '| géométries valides :', iris.is_valid.all())
pd.crosstab(iris.dep, iris.TYP_IRIS, margins=True)

CODE_IRIS unique : True | géométries valides : True


TYP_IRIS,A,D,H,Z,All
dep,,,,,
75,88,43,861,0,992
92,0,2,613,1,616
93,21,2,590,1,614
94,9,0,516,5,530
All,118,47,2580,7,2752


## 2. Filosofi : valeurs manquantes

`ns` = non significatif (secret statistique), `nd` = non disponible. Aucune imputation.

In [3]:
f = pd.read_csv('../data/raw/filosofi/BASE_TD_FILO_IRIS_2021_DISP.csv', sep=';', dtype=str)
f = f[f.IRIS.str[:2].isin(['75', '92', '93', '94'])]
f['med'] = pd.to_numeric(f.DISP_MED21, errors='coerce')
f.groupby(f.IRIS.str[:2]).agg(iris=('IRIS', 'size'), med_ok=('med', 'count'),
    ns=('DISP_MED21', lambda s: (s == 'ns').sum()), nd=('DISP_MED21', lambda s: (s == 'nd').sum()),
    med_des_medianes=('med', 'median'), min=('med', 'min'), max=('med', 'max'))

,iris,med_ok,ns,nd,med_des_medianes,min,max
IRIS,,,,,,,
75,992,864,85,43,31810.0,14900.0,65140.0
92,615,584,30,1,31875.0,14920.0,62570.0
93,613,568,43,2,18750.0,12550.0,31640.0
94,525,513,12,0,24390.0,14180.0,42670.0


In [4]:
f.groupby([f.IRIS.str[:2], 'DISP_NOTE21']).size().unstack(fill_value=0)  # 0 = RAS, 1 = forte évolution du nb de ménages, 5 = non diffusé (anomalie)

DISP_NOTE21,0,1,5,so
IRIS,,,,
75,842,22,0,128
92,574,10,20,11
93,558,10,3,42
94,507,6,0,12


## 3. Jointure IRIS ↔ Filosofi

In [5]:
m = iris.merge(f[['IRIS', 'DISP_MED21', 'med']], left_on='CODE_IRIS', right_on='IRIS', how='outer', indicator=True)
pd.crosstab(m.CODE_IRIS.fillna(m.IRIS).str[:2], m._merge)

_merge,left_only,both
CODE_IRIS,,
75,0,992
92,1,615
93,1,613
94,5,525


In [6]:
m[m._merge == 'left_only'][['CODE_IRIS', 'NOM_COM', 'TYP_IRIS']]  # communes non irisées : Filosofi au niveau commune uniquement

,CODE_IRIS,NOM_COM,TYP_IRIS
1373,920470000,Marnes-la-Coquette,Z
1770,930150000,Coubron,Z
2222,940010000,Ablon-sur-Seine,Z
2506,940470000,Mandres-les-Roses,Z
2521,940530000,Noiseau,Z
2536,940560000,Périgny,Z
2614,940700000,Santeny,Z


In [7]:
b = m[m._merge == 'both']
pd.crosstab(b.dep, [b.TYP_IRIS, b.med.isna()], margins=True)  # médiane manquante par type d'IRIS

TYP_IRIS     A          D          H        All
med      False True False True False True      
dep                                            
75          12   76     0   43   852    9   992
92           0    0     1    1   583   30   615
93           4   17     0    2   564   26   613
94           2    7     0    0   511    5   525
All         18  100     1   46  2510   70  2745